In [0]:
%sql
-- ====================================================================
-- 1. Volumen de procesamiento por capa
-- ====================================================================
SELECT
  CASE
    WHEN origin.flow_name LIKE '%bronze_%' THEN '1. Bronze'
    WHEN origin.flow_name LIKE '%silver_%' THEN '2. Silver'
    WHEN origin.flow_name LIKE '%gold_%' THEN '3. Gold'
    ELSE 'Otro'
  END AS capa,
  SUM(COALESCE(CAST(details:flow_progress:metrics:num_output_rows AS BIGINT), 0)) AS registros_exitosos
FROM fintech_finpay.observability.finpay_event_log_dev
WHERE event_type = 'flow_progress'
GROUP BY 1
HAVING capa != 'Otro'
ORDER BY 1 ASC;

In [0]:
%sql
-- ====================================================================
-- 2. Registros rechazados por reglas de calidad (Expectations)
-- ====================================================================
SELECT 
  origin.flow_name AS tabla_destino,
  SUM(CAST(details:flow_progress:data_quality:dropped_records AS BIGINT)) AS total_rechazados
FROM fintech_finpay.observability.finpay_event_log_dev
WHERE event_type = 'flow_progress'
  AND CAST(details:flow_progress:data_quality:dropped_records AS BIGINT) > 0
GROUP BY 1
ORDER BY total_rechazados DESC;

In [0]:
%sql
-- ====================================================================
-- 3. Resumen de incidentes en Cuarentena (Reto 2)
-- ====================================================================
SELECT
  fuente_origen,
  motivo_rechazo,
  COUNT(*) AS cantidad_registros,
  MAX(fecha_procesamiento) AS ultimo_incidente
FROM fintech_finpay.silver.quarantine
GROUP BY 1, 2
ORDER BY cantidad_registros DESC;

In [0]:
%sql
-- ====================================================================
-- 4. Tendencia diaria de ingesta
-- ====================================================================
SELECT 
  DATE(timestamp) AS fecha_ejecucion,
  origin.flow_name AS tabla,
  SUM(COALESCE(CAST(details:flow_progress:metrics:num_output_rows AS BIGINT), 0)) AS filas_procesadas
FROM fintech_finpay.observability.finpay_event_log_dev
WHERE event_type = 'flow_progress'
  AND origin.flow_name IS NOT NULL
GROUP BY 1, 2
ORDER BY 1 DESC, 3 DESC;

In [0]:
%sql
-- ====================================================================
-- 5. Tasa de éxito del pipeline (Health Score)
-- ====================================================================
SELECT 
  ROUND(
    (SUM(CAST(details:flow_progress:metrics:num_output_rows AS BIGINT)) * 100.0) / 
    NULLIF(
      SUM(CAST(details:flow_progress:metrics:num_output_rows AS BIGINT)) + 
      SUM(CAST(details:flow_progress:data_quality:dropped_records AS BIGINT)), 0
    )
  , 2) AS tasa_exito_pct
FROM fintech_finpay.observability.finpay_event_log_dev
WHERE event_type = 'flow_progress';